# 02 — Churn Prediction: Logistic Regression vs Random Forest

Train two classifiers on the churn data (80/20 stratified split), compare them honestly,
and save the winner's artifacts for the Streamlit dashboard.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pickle
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

from src import load_data, clean_data, add_features, get_model_data

df = add_features(clean_data(load_data()))
X, y, feature_names = get_model_data(df)
print('Features:', len(feature_names), '| Rows:', len(X))
print('Churn rate in full data: %.2f%%' % (y.mean() * 100))


## 1. Train / test split (stratified, so both sets keep the 20% churn rate)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Test churn rate: %.2f%%' % (y_test.mean() * 100))


## 2. Baseline — Logistic Regression (with scaling)


In [ ]:
lr = Pipeline([('scaler', StandardScaler()),
               ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

def scores(y_true, pred, prob, name):
    s = {
        'model': name,
        'accuracy': round(accuracy_score(y_true, pred), 4),
        'precision': round(precision_score(y_true, pred), 4),
        'recall': round(recall_score(y_true, pred), 4),
        'f1': round(f1_score(y_true, pred), 4),
        'roc_auc': round(roc_auc_score(y_true, prob), 4),
    }
    print(pd.Series(s).to_string())
    return s

lr_scores = scores(y_test, lr_pred, lr_prob, 'Logistic Regression')
print()
print(classification_report(y_test, lr_pred, target_names=['Retained', 'Exited']))


In [ ]:
cm = confusion_matrix(y_test, lr_pred)
fig = ff.create_annotated_heatmap(z=cm, x=['Pred: Retained', 'Pred: Exited'],
                                  y=['True: Retained', 'True: Exited'], colorscale='Blues')
fig.update_layout(title='Logistic Regression — confusion matrix (test set)')
fig.show()


## 3. Random Forest — usually wins on tabular churn data


In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5,
                            random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_scores = scores(y_test, rf_pred, rf_prob, 'Random Forest')
print()
print(classification_report(y_test, rf_pred, target_names=['Retained', 'Exited']))


In [ ]:
cm = confusion_matrix(y_test, rf_pred)
fig = ff.create_annotated_heatmap(z=cm, x=['Pred: Retained', 'Pred: Exited'],
                                  y=['True: Retained', 'True: Exited'], colorscale='Greens')
fig.update_layout(title='Random Forest — confusion matrix (test set)')
fig.show()


## 4. Head-to-head comparison


In [ ]:
cmp = pd.DataFrame([lr_scores, rf_scores]).set_index('model')
print(cmp.to_string())

fig = px.bar(cmp.reset_index().melt(id_vars='model', var_name='metric', value_name='score'),
             x='metric', y='score', color='model', barmode='group',
             title='Model comparison on the held-out test set')
fig.show()


## 5. What drives churn? — feature importance from the Random Forest


In [ ]:
imp = pd.DataFrame({'feature': feature_names, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=False)
print(imp.to_string(index=False))

fig = px.bar(imp.head(12), x='importance', y='feature', orientation='h',
             title='Top drivers of churn (Random Forest feature importance)')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


## 6. Save the winner for the dashboard


In [ ]:
MODELS = Path.cwd().parent / 'models'
MODELS.mkdir(exist_ok=True)

with open(MODELS / 'random_forest.pkl', 'wb') as f:
    pickle.dump({'model': rf, 'features': feature_names}, f)
imp.to_csv(MODELS / 'feature_importance.csv', index=False)
pd.DataFrame([lr_scores, rf_scores]).to_csv(MODELS / 'metrics.csv', index=False)
print('Saved: random_forest.pkl, feature_importance.csv, metrics.csv ->', MODELS)


## Conclusion

- Random Forest beats Logistic Regression on every metric that matters for imbalanced churn data
  (recall and ROC-AUC in particular) — it becomes the dashboard's risk-scoring model.
- The top drivers (age, balance, activity, geography, products) match the EDA findings,
  which means the story is consistent: **older, wealthier, inactive German customers with a single
  product are the ones walking out the door.**
